In [ ]:
# Kurva Eksperimen 2: metrik vs jumlah pasien latih.
# Semua angka dibaca langsung dari berkas fold di results/experiment2 (tidak ada angka yang diketik tangan).
# n = 34 adalah jumlah pasien latih maksimum pada LOSO 35 pasien (sama dengan konfigurasi Eksperimen 1).
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fogbench").is_dir())
sys.path.insert(0, str(ROOT))
from fogbench.report import summarize

OUT = ROOT / "kurva_pasien"
PASIEN = [5, 10, 20, 34]
MODEL = {  # label: (folder, warna, penanda) - urutan dan warna sama dengan figur paper
    "CNN-BiLSTM-Attention": ("cnn_bilstm", "#2a78d6", "o"),
    "DeepFoG": ("deep_fog", "#eb6834", "s"),
    "TCN Yang 2024": ("tcn", "#1baf7a", "^"),
}
METRIK = {  # nama berkas: (label sumbu, fungsi pengambil nilai)
    "auroc": ("AUROC (gabungan)", lambda s: s["pooled"]["auroc"]),
    "auprc": ("AUPRC (gabungan)", lambda s: s["pooled"]["auprc"]),
    "sensitivitas": ("Sensitivitas", lambda s: s["pooled"]["sensitivitas"]),
    "spesifisitas": ("Spesifisitas", lambda s: s["pooled"]["spesifisitas"]),
    "f1": ("F1", lambda s: s["pooled"]["f1"]),
    "f1_episode": ("F1 episode", lambda s: s["pooled"]["episode_f1"]),
    "auroc_per_pasien": ("AUROC per pasien (rerata ± SE)", None),
    "waktu_latih_s": ("Waktu latih total 35 fold (s)", lambda s: s["train_time_s_total"]),
    "waktu_inferensi_ms": ("Waktu inferensi (ms, median)", lambda s: s["infer_time_ms_median"]),
}
CATATAN_WAKTU = ("Catatan: waktu DeepFoG pada Eksperimen 2 tidak sebanding dengan model lain\n"
                 "(sebagian fold dijalankan dengan cuDNN dinonaktifkan dan versi PyTorch berbeda).")

ringkas = {(m, n): summarize(ROOT / "results" / "experiment2" / f"models_{d}_train{n}")
           for m, (d, _, _) in MODEL.items() for n in PASIEN}

for berkas, (label, ambil) in METRIK.items():
    fig, ax = plt.subplots(figsize=(7, 4.5))
    for m, (_, warna, penanda) in MODEL.items():
        if ambil is None:
            nilai, se = [], []
            for n in PASIEN:
                a = np.array([p["auroc"] for p in ringkas[(m, n)]["per_subject"]])
                nilai.append(a.mean()); se.append(a.std(ddof=1) / np.sqrt(len(a)))
            nilai, se = np.array(nilai), np.array(se)
            ax.fill_between(PASIEN, nilai - se, nilai + se, color=warna, alpha=0.12, linewidth=0)
        else:
            nilai = [ambil(ringkas[(m, n)]) for n in PASIEN]
        ax.plot(PASIEN, nilai, color=warna, marker=penanda, markersize=7, linewidth=2, label=m)
    ax.set_xlabel("Jumlah pasien latih")
    ax.set_ylabel(label)
    ax.set_title(f"{label} vs. jumlah pasien latih (Eksperimen 2, LOSO 35 fold)", fontsize=10)
    ax.set_xticks(PASIEN)
    ax.grid(axis="y", alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False)
    if berkas.startswith("waktu"):
        fig.text(0.01, 0.01, CATATAN_WAKTU, fontsize=7, color="#555555", va="bottom")
        fig.tight_layout(rect=(0, 0.08, 1, 1))
    else:
        fig.tight_layout()
    fig.savefig(OUT / f"{berkas}.png", dpi=300)
    plt.show()
    plt.close(fig)
    print("tersimpan:", OUT / f"{berkas}.png")
